# OCR sur une véritable image de document téléchargée

Télécharger une image de test publiée par le projet pytesseract, afficher les pixels, transcrire le texte avec Tesseract, puis visualiser les boîtes des mots et exporter texte/CSV/PDF recherchable. **Aucun texte n’est dessiné pour fabriquer l’entrée.** C’est une image de document existante, pas une photographie de scène prise avec un téléphone. Une seconde exécution peut utiliser votre propre photographie.

Source : https://github.com/madmaze/pytesseract/blob/master/tests/data/test.png

Les sorties sont à calculer en exécutant les cellules. Aucun résultat OCR n’est prérempli ou inventé.

## 1. Installation
Dans Colab/Linux seulement, exécuter la cellule apt ci-dessous. Sous Windows installer Tesseract séparément et ajouter le binaire au PATH. Pour une photo réelle, renseigner IMAGE_PATH plus bas.

In [ ]:
%pip install "pytesseract>=0.3.13,<0.4" "pillow>=10,<13" "matplotlib>=3.8,<4"

In [ ]:
import shutil, subprocess, sys
if shutil.which('tesseract') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','tesseract-ocr','fonts-dejavu-core'],check=True)
    else:
        raise RuntimeError('Installer le binaire Tesseract et les données eng avant de continuer.')


In [ ]:
from pathlib import Path
import time, urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
DATA=Path('data'); DATA.mkdir(exist_ok=True)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
def download(url, name):
    path=DATA/name
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return path


## 2. Télécharger et afficher la véritable image
Le téléchargement est vérifié : code HTTP, décodage par Pillow et dimensions. Les fichiers existants sont réutilisés. IMAGE_PATH permet de remplacer le document par votre propre photo. La référence reste facultative et doit être saisie manuellement, jamais dérivée de la sortie OCR.

In [ ]:
from PIL import ImageOps, ImageEnhance, ImageDraw
import pytesseract, csv, json
URL='https://raw.githubusercontent.com/madmaze/pytesseract/master/tests/data/test.png'
IMAGE_PATH=None
path=Path(IMAGE_PATH) if IMAGE_PATH else download(URL,'pytesseract_document.png')
with Image.open(path) as check: check.verify()
image=ImageOps.exif_transpose(Image.open(path)).convert('RGB')
assert image.width>20 and image.height>20
print('Source :', IMAGE_PATH or URL)
print('Dimensions :',image.size,'| octets :',path.stat().st_size)
plt.figure(figsize=(12,8));plt.imshow(image);plt.title('Image réellement téléchargée, avant OCR');plt.axis('off');plt.show()


## 3. Comparer original et prétraitement
On applique niveaux de gris, contraste et agrandissement. Ces opérations ne garantissent pas une amélioration. PSM 6 suppose un bloc uniforme ; PSM 3 convient mieux à une page multibloc.

In [ ]:
gray=ImageOps.grayscale(image)
processed=ImageEnhance.Contrast(gray).enhance(1.5).resize((image.width*2,image.height*2))
texts={}
for name,im in [('original',image),('pretraite',processed)]:
    t=time.perf_counter()
    texts[name]=pytesseract.image_to_string(im,lang='eng',config='--psm 6').strip()
    print(f'--- {name} | {time.perf_counter()-t:.3f} s ---')
    print(texts[name])
assert any(texts.values()), 'Aucun texte reconnu : examiner image, langue et segmentation.'
(OUT/'ocr_original.txt').write_text(texts['original'],encoding='utf-8')
(OUT/'ocr_pretraite.txt').write_text(texts['pretraite'],encoding='utf-8')


## 4. Afficher les mots localisés et leurs scores
Boîtes sur l’image originale. Le score Tesseract est un indicateur interne, pas une probabilité calibrée. Le seuil ne sert qu’à filtrer l’affichage, il ne modifie pas le texte exporté.

In [ ]:
data=pytesseract.image_to_data(image,lang='eng',config='--psm 6',output_type=pytesseract.Output.DICT)
annotated=image.copy();draw=ImageDraw.Draw(annotated);rows=[]
for i,word in enumerate(data['text']):
    if not word.strip():continue
    score=float(data['conf'][i]);x,y,w,h=[int(data[k][i]) for k in ['left','top','width','height']]
    rows.append({'mot':word,'score':score,'x':x,'y':y,'largeur':w,'hauteur':h})
    if score>=40:draw.rectangle((x,y,x+w,y+h),outline='red',width=2)
plt.figure(figsize=(14,9));plt.imshow(annotated);plt.title('Boîtes réellement détectées par Tesseract');plt.axis('off');plt.show()
print('Premiers mots :',rows[:12])
with (OUT/'ocr_mots.csv').open('w',encoding='utf-8',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['mot','score','x','y','largeur','hauteur']);writer.writeheader();writer.writerows(rows)
annotated.save(OUT/'ocr_boites.png')
(OUT/'document_recherchable.pdf').write_bytes(pytesseract.image_to_pdf_or_hocr(image,lang='eng',extension='pdf'))
print('Exports :', [p.name for p in OUT.glob('ocr*')], 'document_recherchable.pdf')


## 5. Évaluation facultative CER/WER
Saisir une transcription humaine exacte du document pour comparer. Sans référence, ne pas prétendre calculer l’accuracy OCR. La normalisation des espaces est commune aux deux textes.

In [ ]:
REFERENCE=None  # coller ici la transcription humaine complète de l’image

def distance(a,b):
    row=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        nxt=[i]
        for j,y in enumerate(b,1):nxt.append(min(nxt[-1]+1,row[j]+1,row[j-1]+(x!=y)))
        row=nxt
    return row[-1]
assert distance('chat','chats')==1
if REFERENCE and REFERENCE.strip():
    ref=' '.join(REFERENCE.split())
    for name,text in texts.items():
        hyp=' '.join(text.split())
        print(name,{'CER':distance(ref,hyp)/len(ref),'WER':distance(ref.split(),hyp.split())/len(ref.split())})
else: print('CER/WER non calculés : aucune transcription humaine fournie.')


## Questions
1. Pourquoi télécharger l’image avant l’OCR ?
2. Pourquoi le prétraitement peut-il dégrader le résultat ?
3. Un score Tesseract suffit-il à calculer le CER ?
4. Que contient le PDF recherchable ?

## Sources et limites
- https://github.com/madmaze/pytesseract/tree/master/tests/data
- https://tesseract-ocr.github.io/tessdoc/
Image de document propre : ne généraliser ni aux manuscrits, ni aux reçus photographiés, ni à toutes les langues. Installer les données `fra` pour le français et changer `lang`. Les droits restent ceux de la source.